#### 1. Importing Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# import opendatasets as od
import pandas as pd
import plotly.express as px
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize'] = (20, 6)
plt.rcParams['figure.facecolor'] = '#00000000'

In [ ]:
# od.version()

#### 2. Loading the Dataset and Initial Exploration

In [ ]:
# download data directly from kaggle
# dataset_url = 'https://www.kaggle.com/jsphyg/weather-dataset-rattle-package'

In [ ]:
import os
# data_dir = './weather-dataset-rattle-package'
# os.listdir(data_dir)
# train_csv = data_dir + '/weatherAUS.csv'
# rain_df = pd.read_csv(train_csv)

In [ ]:
rain_data = pd.read_csv("../data/weatherAUS.csv")
rain_data

As you can see, the dataset contains 145,460 rows and 23 columns. Each row represents a daily weather observation for a specific location in Australia. The columns include various weather attributes such as temperature, humidity, wind speed, and whether it rained the next day (the target variable). The objective here is to create a model that can predict the value in the "RainTomorrow" column based on the other features.

In [ ]:
rain_data.info()

In [ ]:
# columns grouped by data type
rain_data.dtypes.value_counts()

Here is a brief overview of the dataset:
- 23 columns: 7 string columns, 16 numeric columns.
- There are three types of columns:
    - **Date** column: The date of the observation. But we will not be using this column for our model as the day and month of the observation are not related to the conditions of the weather. Rather what matters is the weather conditions of the day, not the name of the day or month.
    - **Numeric** columns: These include features such as `MinTemp`, `MaxTemp`, `Rainfall`, `Evaporation`, `Sunshine`, `WindGustSpeed`, `WindSpeed9am`, `WindSpeed3pm`, `Humidity9am`, `Humidity3pm`, `Pressure9am`, `Pressure3pm`, `Cloud9am`, and `Cloud3pm`.
    - **Categorical** columns: These include `Location`, `WindGustDir`, `WindDir9am`, `WindDir3pm`, `RainToday`, and the target variable `RainTomorrow`.
- The target variable is "RainTomorrow", which indicates whether it rained the next day (Yes/No).

##### Are there any missing values in the dataset?

In [ ]:
rain_data.isnull().sum()

This shows that there are missing values in several columns, with "Sunshine" having the highest number of missing values (69,835) and "MaxTemp" with lowest number of missing values (1,261). The presence of missing values will need to be addressed during the data cleaning and preprocessing stage.

#### 3. Data Cleaning and Preprocessing

In [ ]:
rain_data.dropna(subset=['RainTomorrow', 'RainToday'], inplace=True)

Here I have removed rows with missing values in the "RainToday" and "RainTomorrow" columns, as these are essential for our analysis and modeling. This will help ensure that we have a cleaner dataset to work with for our analysis and modeling. Now our column "RainTomorrow" and "RainToday" will not have any missing values, which is crucial for our target variable and one of the key features in our analysis.
Before: 145,460 rows
After: 140,787 rows
4673 rows with missing values in "RainTomorrow" or "RainToday" have been removed.

#### 4. Exploratory Data Analysis and Visualization

Now let me perform some EDA and visualization to understand the relationship between the features and the target variable "RainTomorrow". I will create some visualizations to explore the distribution of the target variable and how it relates to other features in the dataset. This will help us identify any patterns or trends that may be useful for our modeling.

Let me once see columns there for fast access and then I will start with the visualizations.

In [ ]:
rain_data.info()

In [ ]:
rain_data

### Location vs Rain days (RainToday)

##### Using Seaborn

In [ ]:
sns.histplot(
    data=rain_data,
    x='Location',
    hue='RainToday',
    multiple='stack', # this is 
    palette='Set2'
)
plt.xticks(rotation=90) # Rotate x-axis labels for better readability
plt.title('Distribution of RainToday by Location')
plt.xlabel('Location')
plt.ylabel('Count')
# plt.legend(title='RainToday', labels=['No', 'Yes'])
plt.show()

It is more interactive if I plot it with plotly

In [ ]:
px.histogram(
    data_frame=rain_data,
    x='Location',
    color='RainToday',
    title='Distribution of RainToday by Location',
    # labels={'Location': 'Location', 'count': 'Count'},
    category_orders={'RainToday': ['No', 'Yes']}
)

> From the histogram, I can observe that certain locations have a higher frequency of rain days (RainToday = Yes) compared to others.

### Temp3pm vs RainTomorrow

In [ ]:
px.histogram(
    rain_data,
    x='Temp3pm',
    color='RainTomorrow',
    title='Distribution of Temperature at 3pm by RainTomorrow'
)

### RainTomorrow vs RainToday

In [ ]:
px.histogram(
    rain_data,
    x='RainTomorrow',
    color='RainToday',
    title='Distribution of RainTomorrow by RainToday'
)

### 

### Max_Temp vs Max_Temp

In [ ]:
px.scatter(
    rain_data,
    x = 'MinTemp',
    y = 'MaxTemp',
    color='RainToday',
    title = 'Min temp vs Max temp by RainToday'
)

In [ ]:
sns.histplot(
    rain_data,
    x = 'MinTemp',
    y = 'MaxTemp',
    hue = 'RainToday'
)
plt.title('Min temp vs Max temp by Rain Today')
plt.legend(title='RainToday', labels=['Yes', 'No'])
plt.show()

### Temperature at 3pm vs Humidity at 3pm 

In [ ]:
px.scatter(
    rain_data,
    x='Temp3pm',
    y='Humidity3pm',
    color='RainToday',
    title='Temperature at 3pm vs Humidity at 3pm by RainToday'
)

There so many other missing values in the dataset. So how can I train a model with missing values?
About 60% of the rows have missing values in at least one column. This is a significant portion of the dataset, and it can pose challenges for training a machine learning model.
To handle the missing values, we have a few options:
1. **Remove rows with missing values**: This is the simplest approach, but it can lead to a significant loss of data, especially if a large portion of the dataset has missing values.
2. **Impute missing values**: We can fill in the missing values using various imputation techniques, such as mean, median, mode, or more advanced methods like K-nearest neighbors (KNN) imputation or regression imputation.
3. **Use models that can handle missing values**: Some machine learning algorithms, such as decision trees and random forests, can handle missing values without the need for imputation.
Given the high percentage of missing values, it may be more appropriate to use imputation techniques or models that can handle missing values rather than removing rows, as this would allow us to retain more of the data for training our model.

### Train-Validate-Test

We have to split our dataset into train, validate and test data.
- **Train dataset** is used to train the model and learn the patterns in the data. Here I used $60\%$ of the dataset for training.
- **Validate dataset** is used to tune the hyperparameters of the model and evaluate its performance during the training process. Here I used $20\%$ of the dataset for validation.
- **Test dataset** is used to evaluate the final performance of the model after training and hyperparameter tuning. It provides an unbiased estimate of how well the model will perform on unseen data. Here I used $20\%$ of dataset for testing.


In [ ]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(rain_data, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)

In [ ]:
print('train_df.shape: ', train_df.shape)
print('val_df.shape', val_df.shape)
print('test_df.shape', test_df.shape)

Verify: $90,103 + 22,526 + 28,158 = 140,787$

But as there is date in our above dataset, we have to split the data in a way that the training data comes before the **validation** and **test** data. This is important to prevent data leakage and ensure that our model is trained on past data and evaluated on future data.

In [ ]:
year = pd.to_datetime(rain_data.Date).dt.year
sns.countplot(
    x = year
)
plt.title('Number of Rows per Year')
plt.show()

In [ ]:
# count plot in plotly
fig = px.histogram(
    data_frame=rain_data,
    x = year,
    labels={'x': 'Years', 'count': 'Row Count'},
    title='Number of Rows per Year',
)
fig.update_layout(bargap=0.2)
fig.show()

In [ ]:
# count of rows per year
year.value_counts()

Above is the count of rows in each year.

Now let me update the train, validate and test datasets so that they contain data based on the year.\
**train**: for dataset before 2015 \
**validate**: for dataset of 2025 \
**test**: for dataset after 2025

In [ ]:
train_df = rain_data[year < 2015]
validate_df = rain_data[year == 2015]
test_df = rain_data[year > 2015]


In [ ]:
print(train_df.shape)
print(validate_df.shape)
print(test_df.shape)

Verify: $97,988 + 17,089 + 25,710 = 140,787$

In [ ]:
train_df

In [ ]:
validate_df

In [ ]:
test_df

#### Input dataset and target variable

Not all columns are useful for training a model. In this case, the "Date" column may not be directly useful for training a model, as it is a temporal feature. However, we can extract useful features from the "Date" column, such as the year, month, day of the week, etc., which may help in improving the model's performance.

1. Extract input columns and target variable/column

In [ ]:
input_cols = list(rain_data.columns)[1:-1] # Date and RainTomorrow columns are not included
target_col = 'RainTomorrow'

In [ ]:
input_cols

In [ ]:
target_col

2. Extraction of input and output datasets

train data set

In [ ]:
train_input = train_df[input_cols].copy()
# train_target = (train_df[input_cols[-1]]).copy()
train_target = train_df[target_col].copy()

validation data set

In [ ]:
val_input = validate_df[input_cols].copy()
val_target = validate_df[target_col].copy()

test data set

In [ ]:
test_input = test_df[input_cols].copy()
test_target = test_df[target_col].copy()

**Now, let me put them one by one**

In [ ]:
train_input

In [ ]:
train_target

In [ ]:
val_input

In [ ]:
val_target

In [ ]:
test_input

In [ ]:
test_target

Let me check the rows whether there is any row where MinTem is greater than MaxTemp. If there is such row, then it is an error and we have to remove it from the dataset.

In [ ]:
check = train_input['MinTemp'] > train_input['MaxTemp']
train_input[check]

Here I found 6 rows where MinTemp is equal to MaxTemp.
Would this be an error? It is possible that there are days where the minimum and maximum temperatures are the same, especially if the temperature is constant throughout the day. However, it is also possible that these rows contain errors or outliers. To determine whether these rows should be removed, we can further investigate the data and check for any other anomalies or inconsistencies in those rows. If we find that these rows are indeed errors, we can consider removing them from the dataset to improve the quality of our data for modeling.

#### Identifying Input and Target columns

Let me focus on the training dataset for now. As it is used for training the model, it is important to know which columns are numeric and which are categorical, as this will affect how we preprocess the data and which algorithms we can use for modeling.

In [ ]:
numeric_cols = train_input.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_input.select_dtypes(exclude=np.number).columns.tolist()

Statistics of the numeric columns in the training dataset:

In [ ]:
train_input[numeric_cols].describe().transpose()
# train_input[numeric_cols].describe().T

Which categories are there in the categorical columns of the training dataset?

In [ ]:
train_input[categorical_cols].nunique()

#### Imputation of missing values

Before imputation, let me check how many numeric missing values there are.

In [ ]:
# in whole dataset
rain_data[numeric_cols].isna().sum()

In [ ]:
# in training dataset
train_input[numeric_cols].isna().sum()

Now let me impute the missing values using `SimpleImputer` with the strategy of mean imputation for numeric columns and mode imputation for categorical columns.

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
imputer.fit(rain_data[numeric_cols])

In [ ]:
rain_data['MinTemp'].mean()

In [ ]:
print(imputer.statistics_)

Now I can impute the missing values in the numeric columns of train, validate and test datasets using the fitted imputer with transform method which is already fitted on the rain_df (whole dataset).
Why don't I fit the imputer again for each subset data?
The reason we fit the imputer on the whole dataset (rain_df) and then use the same fitted imputer to transform the train, validate, and test datasets is to ensure that the imputation is consistent across all subsets of the data. By fitting the imputer on the entire dataset, we are calculating the mean (or mode) values based on all available data, which provides a more accurate representation of the missing values.

In [ ]:
train_input[numeric_cols] = imputer.transform(train_input[numeric_cols])
val_input[numeric_cols] = imputer.transform(val_input[numeric_cols])
test_input[numeric_cols] = imputer.transform(test_input[numeric_cols])

In [ ]:
train_input[numeric_cols].isna().sum()

This shows there are no missing values in the numeric columns of the train dataset after imputation. This is the case in the other datasets too.

#### Scaling the numeric columns

Scaling the numeric columns is an important step in the preprocessing of data for machine learning models. It helps to ensure that all features are on the same scale, which can improve the performance of certain algorithms, such as those that rely on distance metrics (e.g., K-nearest neighbors, support vector machines) or gradient-based optimization (e.g., logistic regression, neural networks).

Numeric columns can have different ranges and units, which can lead to issues during model training. For example, if one feature has a much larger range than another, it may dominate the learning process and lead to suboptimal performance. Scaling the numeric columns helps to mitigate this issue by transforming the features to a common scale, such as standardization (mean=0, standard deviation=1) or normalization (scaling to a range of [0, 1]).
In this dataset, there are varying ranges of values.

In [ ]:
train_input[numeric_cols].describe()

For example, the "MinTemp" column has a mean of **12.008** and a standard deviation of **6.337**, while the "WindGustSpeed" column has a mean of **40.19877** and a standard deviation of **13.21224**. This indicates that the "WindGustSpeed" column has a much larger range of values compared to the "MinTemp" column. If we were to train a model without scaling, the "WindGustSpeed" feature may dominate the learning process and lead to suboptimal performance. Therefore, it is important to scale the numeric columns to ensure that all features contribute equally to the model training process.

Now I will use `MinMaxScaler` to scale the numeric columns in the train, validate and test datasets. This will transform the features to a common scale, which can help improve the performance of our machine learning models.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(rain_data[numeric_cols])

Now let me look what the minimum and maximum values of the numeric columns are after scaling.

In [ ]:
print("Minimum: ", scaler.data_min_)

# how to print out only numeric, without data types


In [ ]:
print("Maximum: ", scaler.data_max_)

Now I can separately scale train, validate and test datasets using `transform` method of the fitted scaler

In [ ]:
train_input[numeric_cols] = scaler.transform(train_input[numeric_cols])
val_input[numeric_cols] = scaler.transform(val_input[numeric_cols])
test_input[numeric_cols] = scaler.transform(test_input[numeric_cols])

Now let me verify that values of each column are in the range of [0, 1].

In [ ]:
train_input[numeric_cols].describe()

#### Encoding the Categorical Columns

As machine learning models typically require numeric input, we need to encode the categorical columns in our dataset. This involves converting the categorical variables into a format that can be provided to machine learning algorithms. Common encoding techniques include **one-hot encoding**, **label encoding**, and **ordinal encoding**. 
- **One-hot encoding** creates new binary columns for each category in the categorical variables. This allows the model to learn from the presence or absence of each category without assuming any ordinal relationship between them. One-hot encoding is particularly useful when dealing with nominal categorical variables, where there is no inherent order or ranking among the categories.

- **Label encoding** assigns a unique integer to each category in the categorical variable. This can be useful for ordinal categorical variables, where there is a natural order or ranking among the categories. However, it can introduce unintended ordinal relationships if used with nominal categorical variables.
- **Ordinal encoding** assigns a unique integer to each category in the categorical variable based on a specified order. This is suitable for ordinal categorical variables, where there is a natural order or ranking among the categories. However, it should not be used for nominal categorical variables, as it can introduce unintended ordinal relationships.
The choice of encoding method depends on the nature of the categorical variable and the specific requirements of the machine learning model being used.

Now here I will use **one-hot encoding** which creates new binary columns for each category in the categorical variables. This allows the model to learn from the presence or absence of each category without assuming any ordinal relationship between them. One-hot encoding is particularly useful when dealing with nominal categorical variables, where there is no inherent order or ranking among the categories.

In [ ]:
rain_data[categorical_cols].nunique()

I can perform one-hot encoding using `OneHotEncoder` class from `sklearn.preprocessing`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(rain_data[categorical_cols])

In [ ]:
print(encoder.categories_)

The encoder has created list of categories for each of categorical columns of dataset.

In [ ]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))
encoded_cols

The above are column names for each categorical column.

**Now let me add those columns to train, validate and test datasets.**

In [ ]:
train_input[encoded_cols] = encoder.transform(train_input[categorical_cols]).copy()

In [ ]:
val_input[encoded_cols] = encoder.transform(val_input[categorical_cols]).copy()

In [ ]:
test_input[encoded_cols] = encoder.transform(test_input[categorical_cols]).copy()

Now let me check that these columns are added to train, validate and test datasets. Now each set contains 49 + 16 + 16 + 16 + 2 +

In [ ]:
train_input

In [ ]:
val_input

In [ ]:
test_input

In [ ]:
train_input[categorical_cols].nunique()

### Saving processed Data to Disk
- Now preprocessing of data is finished. Data are ready to train the model because there are no missing values, numerical values are scaled and all categorical columns are encoded.

Let me once the shape of datasets before saving.

In [ ]:
print('train inputs: ', train_input.shape)
print('train_target: ', train_target.shape)
print('val_inputs: ', val_input.shape)
print('val_target: ', val_target.shape)
print('test_inputs: ', test_input.shape)
print('test_target: ', test_target.shape)

# 97,988 + 17,089 + 25,710 = 140,787

Now I can save to disk. I will use **parquet** format for saving because it is fast and efficient format for saving Pandas dataframes.
> For `parquet`, install **payrrow** using pip: `pip install pandas pyarrow`

In [ ]:
train_input.to_parquet('../data/train_dataset/train_inputs.parquet')
val_input.to_parquet('../data/validate_dataset/val_inputs.parquet')
test_input.to_parquet('../data/test_dataset/test_inputs.parquet')

In [ ]:
pd.DataFrame(train_target).to_parquet('../data/train_dataset/train_target.parquet')
pd.DataFrame(val_target).to_parquet('../data/validate_dataset/val_target.parquet')
pd.DataFrame(test_target).to_parquet('../data/test_dataset/test_target.parquet')

**Verify that datasets are loaded successfully**

In [ ]:
train_inputs = pd.read_parquet('../data/train_dataset/train_inputs.parquet')
val_inputs = pd.read_parquet('../data/validate_dataset/val_inputs.parquet')
test_inputs = pd.read_parquet('../data/test_dataset/test_inputs.parquet')

train_target = pd.read_parquet('../data/train_dataset/train_target.parquet')[target_col]
val_target = pd.read_parquet('../data/validate_dataset/val_target.parquet')[target_col]
test_target = pd.read_parquet('../data/test_dataset/test_target.parquet')[target_col]


In [ ]:
print('train inputs: ', train_inputs.shape)
print('train_target: ', train_target.shape)

print('val_inputs: ', val_input.shape)
print('val_target: ', val_target.shape)

print('test_inputs: ', test_input.shape)
print('test_target: ', test_target.shape)

# 97,988 + 17,089 + 25,710 = 140,787

In [ ]:
train_inputs

In [ ]:
train_target

**_Now everything is ready, I can start training the model!_**

# Training Logistic Regression Model

Now I can train a Logistic Regression classifier.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

log_reg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])


In [ ]:
log_reg_model.fit(train_input, train_target)


#### Model Evaluation

Accuracy alone can be misleading because rainy days are less frequent than non-rainy days, so I will also check precision, recall, F1 score, ROC AUC, and the confusion matrix.


In [ ]:
def evaluate_model(model, inputs, target, split_name):
    predictions = model.predict(inputs)
    yes_index = list(model.named_steps['model'].classes_).index('Yes')
    probabilities = model.predict_proba(inputs)[:, yes_index]

    metrics = {
        'split': split_name,
        'accuracy': accuracy_score(target, predictions),
        'precision': precision_score(target, predictions, pos_label='Yes'),
        'recall': recall_score(target, predictions, pos_label='Yes'),
        'f1': f1_score(target, predictions, pos_label='Yes'),
        'roc_auc': roc_auc_score((target == 'Yes').astype(int), probabilities),
    }

    print(f"{split_name} classification report")
    print(classification_report(target, predictions))
    return metrics

train_metrics = evaluate_model(log_reg_model, train_input, train_target, 'Train')
val_metrics = evaluate_model(log_reg_model, val_input, val_target, 'Validation')
test_metrics = evaluate_model(log_reg_model, test_input, test_target, 'Test')

pd.DataFrame([train_metrics, val_metrics, test_metrics]).set_index('split')


In [ ]:
test_predictions = log_reg_model.predict(test_input)
cm = confusion_matrix(test_target, test_predictions, labels=['No', 'Yes'])

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes']).plot(cmap='Blues')
plt.title('Logistic Regression Confusion Matrix - Test Set')
plt.show()


#### Make Predictions

The trained pipeline can now preprocess new weather rows and return both the predicted class and the probability of rain tomorrow.


In [ ]:
sample_inputs = test_input.head(5)
sample_predictions = log_reg_model.predict(sample_inputs)
yes_index = list(log_reg_model.named_steps['model'].classes_).index('Yes')
sample_probabilities = log_reg_model.predict_proba(sample_inputs)[:, yes_index]

pd.DataFrame({
    'Actual': test_target.head(5).values,
    'Predicted': sample_predictions,
    'RainTomorrow_Probability': sample_probabilities.round(3)
})
